In [2]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, 17e78596-d33a-4353-ac17-4d5f76eec576, 4, Finished, Available, Finished, False)

# dim location

In [3]:
# Your fixed source/destination points, with names for readable dashboards.
# location_id is a stable surrogate key derived from rounded lat/long.

sources_data = [
    (23.0211, 72.5606, "Parimal Garden"),
    (23.0264, 72.5621, "Law Garden Night Market"),
    (23.0371, 72.5532, "Amdavad ni Gufa"),
    (23.0105, 72.5714, "Sanskar Kendra"),
    (23.0374, 72.5535, "Zen Cafe"),
    (23.0135, 72.5721, "Riverfront Flower Park"),
    (23.0075, 72.6042, "Kamla Nehru Zoo"),
    (23.0345, 72.5484, "L.D. College of Engineering"),
    (23.0336, 72.5458, "Gujarat University"),
    (23.0311, 72.5412, "Physical Research Laboratory (PRL)"),
    (23.0116, 72.5671, "National Institute of Design (NID)"),
    (23.0315, 72.5435, "ATIRA"),
    (23.0298, 72.6011, "Kalupur Railway Station"),
    (23.0161, 72.5936, "Geeta Mandir Bus Terminus"),
    (23.0197, 72.5448, "Nehrunagar Circle"),
    (23.0426, 72.5712, "Income Tax Circle"),
    (23.0232, 72.5728, "Ellisbridge (Old)"),
    (23.0145, 72.5615, "Paldi Crossroad"),
    (23.0398, 72.5881, "Delhi Darwaja"),
    (23.0353, 72.5292, "Vastrapur Lake Park"),
]

destinations_data = [
    (23.0600, 72.5808, "Sabarmati Ashram"),
    (23.1667, 72.5801, "Adalaj Stepwell"),
    (23.0062, 72.6015, "Kankaria Lake"),
    (23.0270, 72.5811, "Sidi Saiyyed Mosque"),
    (23.0248, 72.5872, "Jama Masjid"),
    (23.0253, 72.5800, "Bhadra Fort"),
    (23.0258, 72.5886, "Manek Chowk"),
    (23.0381, 72.5919, "Hutheesing Jain Temple"),
    (23.0506, 72.5925, "Calico Museum of Textiles"),
    (22.9818, 72.5034, "Sarkhej Roza"),
]

# pickup location table
dim_pickup_location = spark.createDataFrame(
    sources_data, ["latitude", "longitude", "place_name"]
)

dim_pickup_location = dim_pickup_location.withColumn(
    "location_id",
    F.sha2(F.concat_ws("_", F.col("latitude"), F.col("longitude")), 256)
).withColumn("ingested_at", F.current_timestamp())

dim_pickup_location.write.mode("overwrite").saveAsTable("gold.dim_pickup_location")
display(dim_pickup_location)

# destination location table
dim_destination_location = spark.createDataFrame(
    destinations_data, ["latitude", "longitude", "place_name"]
)

dim_destination_location = dim_destination_location.withColumn(
    "location_id",
    F.sha2(F.concat_ws("_", F.col("latitude"), F.col("longitude")), 256)
).withColumn("ingested_at", F.current_timestamp())

dim_destination_location.write.mode("overwrite").saveAsTable("gold.dim_destination_location")
display(dim_destination_location)

StatementMeta(, 5695e2c6-d16f-427f-ab6f-222e35afe740, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 493f8b32-a9a3-43d4-9b33-b817108a55b5)

SynapseWidget(Synapse.DataFrame, f2eebcba-2799-4cf2-b900-91a0ecce4295)

# dim driver (join with vehicle)

In [3]:
driver_silver = spark.read.table("silver.driver")
vehicle_silver = spark.read.table("silver.vehicle")

dim_driver = (
    driver_silver.alias("d")
    .join(vehicle_silver.alias("v"), F.col("d.driver_id") == F.col("v.driver_id"), "left")
    .select(
        F.col("d.driver_id"),
        F.col("d.name"),
        F.col("d.gender"),
        F.col("d.rating"),
        F.col("d.is_active"),
        F.col("d.created_at").alias("driver_since"),
        F.col("v.vehicle_type"),
        F.col("v.model"),
        F.col("v.color"),
        F.col("v.rc_verified"),
    )
    .withColumn("ingested_at", F.current_timestamp())
)

dim_driver.write.mode("overwrite").saveAsTable("gold.dim_driver")
display(dim_driver.limit(10))

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ca35875-c7e4-400c-b0c7-73ed91f84b3b)

# dim rider

In [4]:
rider_silver = spark.read.table("silver.rider")

dim_rider = (
    rider_silver
    .select("rider_id", "name", "gender", "created_at")
    .withColumnRenamed("created_at", "rider_since")
    .withColumn("ingested_at", F.current_timestamp())
)

dim_rider.write.mode("overwrite").saveAsTable("gold.dim_rider")
display(dim_rider.limit(10))

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b58a87f-c20b-4cee-92c4-1bb3d59ad81d)

# dim date

In [5]:
# Build a date dimension covering your full simulation range.
# Adjust start_date if your data goes back further.

date_range = spark.sql("""
    SELECT explode(sequence(
        to_date('2026-01-01'),
        current_date(),
        interval 1 day
    )) AS full_date
""")

dim_date = (
    date_range
    .withColumn("date_id", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .withColumn("ingested_at", F.current_timestamp())
)

dim_date.write.mode("overwrite").saveAsTable("gold.dim_date")
display(dim_date.limit(10))

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b01e9ba1-a356-435b-945e-d45e912292ce)

# Fact trips

In [6]:
trips_terminal = spark.read.table("silver.trips_terminal")
loc = spark.read.table("gold.dim_location").select(
    "location_id", "latitude", "longitude"
)

# match pickup coords to dim_location
fact_trips = (
    trips_terminal.alias("t")
    .join(
        loc.alias("p"),
        (F.round(F.col("t.pickup_lat"), 4) == F.round(F.col("p.latitude"), 4)) &
        (F.round(F.col("t.pickup_long"), 4) == F.round(F.col("p.longitude"), 4)),
        "left"
    )
    .withColumnRenamed("location_id", "pickup_location_id")
    .drop("latitude", "longitude")
)

fact_trips = (
    fact_trips.alias("t")
    .join(
        loc.alias("d"),
        (F.round(F.col("t.destination_lat"), 4) == F.round(F.col("d.latitude"), 4)) &
        (F.round(F.col("t.destination_long"), 4) == F.round(F.col("d.longitude"), 4)),
        "left"
    )
    .withColumnRenamed("location_id", "destination_location_id")
    .drop("latitude", "longitude")
)

fact_trips = (
    fact_trips
    .withColumn("date_id", F.date_format("requested_at", "yyyyMMdd").cast("int"))
    .select(
        "trip_id", "rider_id", "driver_id",
        "pickup_location_id", "destination_location_id", "date_id",
        "status", "fare_to_rider", "profit_to_driver",
        "distance_km", "duration_minutes",
        "requested_at", "confirmed_at", "started_at", "completed_at",
        "ingested_at"
    )
)

fact_trips.write.mode("overwrite").saveAsTable("gold.fact_trips")
display(fact_trips.limit(10))
print(f"fact_trips rows: {fact_trips.count()}")

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2796363b-092e-4719-965e-29f93412c493)

fact_trips rows: 4


# Fact payments

In [7]:
payments_silver = spark.read.table("silver.payments")
trips_for_date = spark.read.table("gold.fact_trips").select("trip_id", "date_id")

fact_payments = (
    payments_silver.alias("p")
    .join(trips_for_date.alias("t"), "trip_id", "left")
    .select(
        "payment_id", "trip_id", "date_id",
        "amount", "method", "status", "paid_at",
        "ingested_at"
    )
)

fact_payments.write.mode("overwrite").saveAsTable("gold.fact_payments")
display(fact_payments.limit(10))

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 311c9eb5-d2a4-4ba1-86e1-95704b594394)

# Fact trip funnel

In [3]:
trip_transitions = spark.read.table("silver.trip_transitions")

fact_trip_funnel = (
    trip_transitions
    # .filter(F.col("prev_event_type").isNotNull())
    .withColumn("date_id", F.date_format("event_time", "yyyyMMdd").cast("int"))
    .select(
        "trip_id", "date_id",
        "prev_event_type", "event_type",
        "seconds_in_prev_state", "event_time",
        "ingested_at"
    )
)

fact_trip_funnel.write.mode("overwrite").saveAsTable("gold.fact_trip_funnel")
display(fact_trip_funnel.limit(10))

StatementMeta(, 17e78596-d33a-4353-ac17-4d5f76eec576, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 828388a6-cc19-47c0-90bf-e2659f022369)

# Fact Driver performance


In [9]:
fact_trips_g = spark.read.table("gold.fact_trips")
ratings_silver = spark.read.table("silver.ratings")

daily_trip_agg = (
    fact_trips_g
    .filter(F.col("status") == "completed")
    .groupBy("driver_id", "date_id")
    .agg(
        F.count("trip_id").alias("trips_completed"),
        F.sum("profit_to_driver").alias("total_earnings"),
        F.avg("distance_km").alias("avg_distance_km"),
    )
)

daily_cancel_agg = (
    fact_trips_g
    .filter(F.col("status") == "cancelled")
    .groupBy("driver_id", "date_id")
    .agg(F.count("trip_id").alias("cancellation_count"))
)

daily_rating_agg = (
    ratings_silver.alias("r")
    .join(fact_trips_g.select("trip_id", "date_id").alias("t"), "trip_id")
    .groupBy("driver_id", "date_id")
    .agg(F.avg("score").alias("avg_rating_that_day"))
)

fact_driver_performance = (
    daily_trip_agg
    .join(daily_cancel_agg, ["driver_id", "date_id"], "outer")
    .join(daily_rating_agg, ["driver_id", "date_id"], "outer")
    .fillna(0, subset=["trips_completed", "total_earnings", "cancellation_count"])
    .withColumn("ingested_at", F.current_timestamp())
)

fact_driver_performance.write.mode("overwrite").saveAsTable("gold.fact_driver_performance")
display(fact_driver_performance.limit(10))

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c61f8ea-7eb4-4cc9-89f6-2e354cfaf741)

In [10]:
print("=== Gold layer row counts ===")
for t in ["dim_location", "dim_driver", "dim_rider", "dim_date",
          "fact_trips", "fact_payments", "fact_trip_funnel", "fact_driver_performance"]:
    cnt = spark.read.table(f"gold.{t}").count()
    print(f"gold.{t}: {cnt}")

print("\n=== fact_trips status breakdown ===")
spark.read.table("gold.fact_trips").groupBy("status").count().show()

StatementMeta(, 2e7b7975-a553-46ed-99c3-2be756693c0d, 12, Finished, Available, Finished, False)

=== Gold layer row counts ===
gold.dim_location: 30
gold.dim_driver: 20
gold.dim_rider: 50
gold.dim_date: 173
gold.fact_trips: 4
gold.fact_payments: 4
gold.fact_trip_funnel: 12
gold.fact_driver_performance: 4

=== fact_trips status breakdown ===
+---------+-----+
|   status|count|
+---------+-----+
|completed|    4|
+---------+-----+

